In [1]:
import os

PROJECT = "/content/warp_reduction"
SRC = os.path.join(PROJECT, "src")
RESULTS = os.path.join(PROJECT, "results")

os.makedirs(SRC, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

In [2]:
%%writefile /content/warp_reduction/src/benchmark.cu
#include <iostream>
#include <vector>
#include <numeric>
#include <cmath>
#include <cuda_runtime.h>

#define CHECK(call) do { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        std::cerr << "CUDA error: " \
                  << cudaGetErrorString(err) << "\n"; \
        return 1; \
    } \
} while (0)

constexpr int N = 256;
constexpr int THREADS = 256;
constexpr int WARPS = THREADS / 32;

__global__ void reduceShared(
    const int* input, int* output
) {
    __shared__ int sdata[THREADS];

    int tid = threadIdx.x;
    sdata[tid] = input[tid];
    __syncthreads();

    for (int stride = THREADS / 2;
         stride > 0; stride /= 2) {
        if (tid < stride) {
            sdata[tid] += sdata[tid + stride];
        }
        __syncthreads();
    }

    if (tid == 0) output[0] = sdata[0];
}

__global__ void reduceShuffle(
    const int* input, int* output
) {
    __shared__ int warpSums[WARPS];

    int tid = threadIdx.x;
    int lane = tid % 32;
    int warp = tid / 32;

    int value = input[tid];

    // Reduce within each warp.
    unsigned int mask = 0xffffffff;
    for (int offset = 16; offset > 0; offset /= 2) {
        value += __shfl_down_sync(
            mask, value, offset
        );
    }

    // One lane writes each warp's result.
    if (lane == 0) {
        warpSums[warp] = value;
    }
    __syncthreads();

    // First warp combines the 8 warp totals.
    if (warp == 0) {
        int total = (lane < WARPS)
            ? warpSums[lane] : 0;

        for (int offset = 16; offset > 0; offset /= 2) {
            total += __shfl_down_sync(
                mask, total, offset
            );
        }

        if (lane == 0) output[0] = total;
    }
}

float benchmark(
    void (*kernel)(const int*, int*),
    const int* d_input,
    int* d_output
) {
    // Not used: CUDA kernel launch syntax cannot be
    // passed as an ordinary function pointer here.
    return 0.0f;
}

int main() {
    std::vector<int> h_input(N);
    for (int i = 0; i < N; i++) {
        h_input[i] = (i % 10) + 1;
    }

    long long expected = std::accumulate(
        h_input.begin(), h_input.end(), 0LL
    );

    int *d_input = nullptr, *d_output = nullptr;
    CHECK(cudaMalloc(&d_input, N * sizeof(int)));
    CHECK(cudaMalloc(&d_output, sizeof(int)));
    CHECK(cudaMemcpy(
        d_input, h_input.data(), N * sizeof(int),
        cudaMemcpyHostToDevice
    ));

    int sharedResult = 0, shuffleResult = 0;

    reduceShared<<<1, THREADS>>>(d_input, d_output);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    CHECK(cudaMemcpy(
        &sharedResult, d_output, sizeof(int),
        cudaMemcpyDeviceToHost
    ));

    reduceShuffle<<<1, THREADS>>>(d_input, d_output);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    CHECK(cudaMemcpy(
        &shuffleResult, d_output, sizeof(int),
        cudaMemcpyDeviceToHost
    ));

    std::cout << "CPU expected: " << expected << "\n";
    std::cout << "Shared result: " << sharedResult << "\n";
    std::cout << "Shuffle result: " << shuffleResult << "\n";

    if (sharedResult != expected ||
        shuffleResult != expected) {
        std::cerr << "Correctness test FAILED\n";
        cudaFree(d_input);
        cudaFree(d_output);
        return 1;
    }
    std::cout << "Correctness test PASSED\n";

    // Warm-up launches.
    for (int i = 0; i < 100; i++) {
        reduceShared<<<1, THREADS>>>(d_input, d_output);
        reduceShuffle<<<1, THREADS>>>(d_input, d_output);
    }
    CHECK(cudaDeviceSynchronize());

    constexpr int ITERS = 1000;
    cudaEvent_t start, stop;
    CHECK(cudaEventCreate(&start));
    CHECK(cudaEventCreate(&stop));

    CHECK(cudaEventRecord(start));
    for (int i = 0; i < ITERS; i++) {
        reduceShared<<<1, THREADS>>>(d_input, d_output);
    }
    CHECK(cudaEventRecord(stop));
    CHECK(cudaEventSynchronize(stop));

    float sharedMs = 0;
    CHECK(cudaEventElapsedTime(&sharedMs, start, stop));

    CHECK(cudaEventRecord(start));
    for (int i = 0; i < ITERS; i++) {
        reduceShuffle<<<1, THREADS>>>(d_input, d_output);
    }
    CHECK(cudaEventRecord(stop));
    CHECK(cudaEventSynchronize(stop));

    float shuffleMs = 0;
    CHECK(cudaEventElapsedTime(&shuffleMs, start, stop));

    std::cout << "Shared avg (us): "
              << sharedMs * 1000.0f / ITERS << "\n";
    std::cout << "Shuffle avg (us): "
              << shuffleMs * 1000.0f / ITERS << "\n";

    CHECK(cudaEventDestroy(start));
    CHECK(cudaEventDestroy(stop));
    CHECK(cudaFree(d_input));
    CHECK(cudaFree(d_output));
    return 0;
}

Writing /content/warp_reduction/src/benchmark.cu


In [3]:
!nvcc /content/warp_reduction/src/benchmark.cu \
    -o /content/warp_reduction/src/benchmark

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [4]:
!/content/warp_reduction/src/benchmark

CPU expected: 1396
Shared result: 1396
Shuffle result: 1396
Correctness test PASSED
Shared avg (us): 3.80928
Shuffle avg (us): 3.34522
